# DeepLabCut 工具箱

以下是一些可能对您有帮助的资源：

- [github.com/DeepLabCut/DeepLabCut](https://github.com/DeepLabCut/DeepLabCut)
- [DeepLabCut 文档：单动物项目用户指南](https://deeplabcut.github.io/DeepLabCut/docs/standardDeepLabCut_UserGuide.html)

本 Jupyter Notebook 演示了在您自己的项目中使用 DeepLabCut 所需的必要步骤。
它展示了最基础的代码实现方式，但许多函数都包含额外的功能，因此请务必查阅其概述以及相关的协议论文！

本 Notebook 阐述了如何执行以下操作：
- 创建项目
- 提取训练帧
- 标注帧
- 绘制已标注图像
- 创建训练数据集
- 训练网络
- 评估网络
- 分析新的视频
- 创建自动标注的视频
- 绘制轨迹

本 Notebook 演示了在您自己的项目中使用 DeepLabCut 所需的必要步骤。

它展示了最基础的代码实现方式，但许多函数都包含额外的功能，因此请务必查阅其概述以及相关的协议论文！

Nath\*, Mathis\* 等人：使用 DeepLabCut 进行跨物种无标记姿态估计。自然方案（Nature Protocols）, 2019 年。

论文：https://www.nature.com/articles/s41596-019-0176-0

预印本：https://www.biorxiv.org/content/biorxiv/early/2018/11/24/476531.full.pdf

## 创建一个新项目

如果你想使用不同的网络来分析数据，最好将项目分开。如果你要追踪相似的主题或项目（即使是在不同的环境中），你应该使用一个项目。此功能会在用户定义的目录中创建一个带有子目录和基本配置文件的项目，否则项目将在当前工作目录中创建。

在项目的任何阶段，你都可以随时向项目中添加新的视频（用于标记更多数据）。

In [ ]:
import deeplabcut

In [ ]:
task = "Reaching" # Enter the name of your experiment Task
experimenter = "Mackenzie" # Enter the name of the experimenter
video = [
    "videos/video1.avi",
    "videos/video2.avi",
] # Enter the paths of your videos OR FOLDER you want to grab frames from.

path_config_file = deeplabcut.create_new_project(
    task,
    experimenter,
    video,
    copy_videos=True,
)

# NOTE: The function returns the path, where your project is. 
# You could also enter this manually (e.g. if the project is already created and you want to pick up, where you stopped...)
# Enter the path of the config file that was just created from the above step (check the folder):
# path_config_file = "/home/Mackenzie/Reaching/config.yaml"

## 现在，去编辑已创建的 `config.yaml` 文件！
添加你的身体部位标签（body part labels），编辑每段视频需要提取的帧数（number of frames to extract per video），等等。

#### 请注意，您可以通过在任何函数名称后添加一个 `?` 来查看关于该函数的更多信息，例如：

In [ ]:
deeplabcut.extract_frames?

## 从视频中提取帧

成功特征检测器的一个关键点是选择多样化的帧，这些帧应能代表您所研究并需要标记的行为。

此函数根据特定视频（或文件夹）**均匀采样**选择 $N$ 帧（选项为 'uniform'）。注意：如果目标行为在视频中分布稀疏，这种均匀采样方式可能无法产生足够多样的帧（可以考虑使用 K-means 聚类方法），或者也可以选择手动选取帧等方式。

此外，请务必从不同（行为）**会话**和不同的**动物**中选取数据（前提是这些数据存在显著差异），以训练出一个具有不变性的特征检测器。

单独的图像不应太大（即应小于 $850 \times 850$ 像素）。虽然这一步可以稍后处理，但建议尽早裁剪（crop）帧，尽可能去除帧中不必要的区域。

始终检查裁剪后的输出结果。如果对结果满意，则继续进行后续的标记工作。

In [ ]:
%matplotlib inline
#there are other ways to grab frames, such as uniformly; please see the paper:

#AUTOMATIC:
deeplabcut.extract_frames(path_config_file) 

## 标记已提取的帧

只有配置文件中的视频才能用于提取帧。每个视频的提取标签存储在项目目录下的 **'labeled-data'** 子目录中。每个子目录都以视频的名称命名。工具箱中有一个可用于标记操作的标注工具箱。

有关标注的更多信息，请查阅我们的 [napari-deeplabcut 文档](https://github.com/DeepLabCut/napari-deeplabcut/tree/main)！

In [ ]:
# napari will pop up!
# Please go to plugin > deeplabcut to start
# then, drag-and-drop the project configuration file into the viewer (the value of path_config_file)
# finally, drop the folder containing the images (in 'labeled-data') in the viewer

%gui qt6
import napari
napari.Viewer()

## 检查标签

[可选] 检查标签是否已正确创建并存储，这对训练过程是有益的，因为**标签标注**是创建训练数据集最关键的步骤之一。DeepLabCut 工具箱提供了一个名为 `check\_labels` 的函数来实现此目的。其用法如下：

In [ ]:
deeplabcut.check_labels(path_config_file) # this creates a subdirectory with the frames + your labels

如果需要调整标签，您可以使用重新启动标注 GUI 的方式来移动它们，保存，然后重新绘图！

## 创建训练数据集

这个函数根据包含标签信息的 `pandas` 数据帧，生成网络训练所需的训练数据信息。用户可以在 `config.yaml` 文件中设置训练集的大小比例（相对于 `hd5` 文件中所有已标记的图像）。在创建数据集时，如果用户想要对性能进行基准测试（Benchmarking），他们可以创建多个不同的数据洗牌（shuffles）（通常情况下，您只会设置一个，所以可以不进行任何操作！）。

运行此脚本后，训练数据集将被创建，并保存在项目目录下的子目录 **`training-datasets`** 中。

此函数还会**自动**在 **`dlc-models-pytorch`** 目录下创建新的子目录，并生成一个 `pytorch_config.yaml` 文件。该文件定义了模型架构，并包含了用于网络训练的各种参数。在大多数我们遇到的使用场景中，默认值通常都是合适的。关于可以设置的变量的更多信息，请查阅 [文档](https://deeplabcut.github.io/DeepLabCut/docs/pytorch/pytorch_config.html)！

现在，是时候开始训练网络了！

In [ ]:
deeplabcut.create_training_dataset(path_config_file)

# remember, there are several networks you can pick, the default is resnet-50!

## 开始训练：

用户可以在 `.../project-name/dlc-models-pytorch/.../pytorch_config.yaml` 文件中设置各种参数。有关可设置变量的更多信息，请参阅 [文档](https://deeplabcut.github.io/DeepLabCut/docs/pytorch/pytorch_config.html)！

此函数会针对训练数据集的特定洗牌（shuffle）来训练网络。

In [ ]:
deeplabcut.train_network(path_config_file)

## 开始评估

此函数用于评估一个已训练的模型，评估对象可以是特定轮次（shuffle/shuffles）下的特定状态，也可以是数据集（images）上的所有状态，并将结果作为 `.csv` 文件存储在 `evaluation-results-pytorch` 目录下的一个子目录中。

In [ ]:
deeplabcut.evaluate_network(path_config_file, plotting=True)

```markdown
## 开始分析视频

此函数用于分析新的视频。用户可以从评估结果中选择最佳模型，并在 `config.yaml` 文件中为变量 **snapshotindex** 指定正确的快照索引（snapshot index）。否则，系统将默认使用最新的快照来分析视频。

分析结果将存储在与视频文件位于同一目录下的 hd5 文件中。
```

In [ ]:
videofile_path = ['videos/video3.avi', 'videos/video4.avi'] # Enter a folder OR a list of videos to analyze.

deeplabcut.analyze_videos(path_config_file,videofile_path, videotype='.avi')

## 提取异常帧 [可选步骤]

这是一个可选步骤，仅在评估结果不佳（即标签预测错误）时使用。在这种情况下，用户可以使用以下函数来提取标签预测不正确的帧。此步骤具有许多选项，请参考：

In [ ]:
deeplabcut.extract_outlier_frames?

In [ ]:
deeplabcut.extract_outlier_frames(path_config_file,['/videos/video3.avi']) #pass a specific video

## 完善标签 [可选步骤]
在提取了异常帧之后，用户可以使用以下函数将预测的标签移动到正确的位置。 **这样做可以增加（或增强）** 训练数据集。

In [ ]:
%gui qt6
deeplabcut.refine_labels(path_config_file)

**注意：** 之后，如果你想查看经过调整的帧（frames），可以通过运行以下命令在主 GUI 中加载它们：

```python
deeplabcut.label_frames(path_config_file)
```

（你可以在下方添加一个新的“单元格”来包含这段代码！）

#### 一旦所有文件夹都重新标注完毕，请再次检查标签！如果你不满意，可以在主 GUI 中进行调整：

```python
deeplabcut.label_frames(path_config_file)
```

检查标签：

```python
deeplabcut.check_labels(path_config_file)
```

In [ ]:
#NOW, merge this with your original data:

deeplabcut.merge_datasets(path_config_file)

## 创建新一轮训练数据集 [可选步骤]
在对标签进行完善并将其附加到原始数据集之后，这就创建了新一轮的训练数据集。这个设置在 `config.yaml` 文件中是自动配置好的，所以让我们马上开始训练吧！

In [ ]:
deeplabcut.create_training_dataset(path_config_file)

## 创建带标签的视频

此函数用于可视化目的，可以用来创建一个 `.mp4` 格式的视频，该视频中包含网络预测的标签。此视频将保存在原始视频所在的相同目录下。

**这个函数有许多有趣的功能选项！**

```python
deeplabcut.create_labeled_video(
    config,
    videos,
    videotype='avi',
    shuffle=1,
    trainingsetindex=0,
    filtered=False,
    save_frames=False,
    Frames2plot=None,
    delete=False,
    displayedbodyparts='all',
    codec='mp4v',
    outputframerate=None,
    destfolder=None,
    draw_skeleton=False,
    trailpoints=0,
    displaycropped=False,
)
```

所以请检查：

In [ ]:
deeplabcut.create_labeled_video?

In [ ]:
deeplabcut.create_labeled_video(path_config_file,videofile_path)

## 绘制已分析视频的轨迹

此函数会绘制整个视频中所有身体部位的轨迹。每个身体部位都会通过唯一的颜色进行标识。

In [ ]:
%matplotlib notebook #for making interactive plots.
deeplabcut.plot_trajectories(path_config_file, videofile_path)